<h1 style="color:black;"> Data Preparation </h1>

This project investigates whether demographic, anthropometric and routinely available clinical variables can identify adults with hepatic steatosis and which variables contribute most strongly to screening. It also examines whether screening performance and influential predictors differ across data-driven metabolic phenotypes. 

This notebook prepares the NHANES 2017-March 2020 pre-pandemic data for subsequent unsupervised clustering and supervised machine learning analyses. 

Variables were selected to represent clinically relevant metabolic domains, including general body size and obesity status, central adiposity, blood pressure, glucose regulation and lipid metabolism. 

The hepatic steatosis outcome is defined later using the median controlled attenuation parameter (NHANES variable `LUXCAPM`, later renamed to `cap_db_m`) obtained through liver transient elastography.

<h2 style="color:black;"> Importing libraries and loading datasets </h2>

This section imports the required libraries and defines the paths for the raw and processed 2017-2020 NHANES data. It also creates the processed-data directory when necessary.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/raw/2017-2020")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

The `files` dictionary maps each NHANES component to its corresponding XPT file. The XPT files are then loaded into pandas DataFrames that are stored in a `data` dictionary for subsequent processing.

In [2]:
files = {
    "demo": "P_DEMO.xpt",
    "body": "P_BMX.xpt",
    "bp": "P_BPXO.xpt",
    "liver": "P_LUX.xpt",
    "hdl": "P_HDL.xpt",
    "triglycerides": "P_TRIGLY.xpt",
    "glucose": "P_GLU.xpt",
    "diabetes": "P_DIQ.xpt",
}

data = {
    name: pd.read_sas(DATA_DIR / filename, format="xport")
    for name, filename in files.items()
}

The dimensions of each DataFrame are displayed to verify successful loading of all component datasets:

In [3]:
for name, df in data.items():
    print(f"{name:15} {df.shape}")

demo            (15560, 29)
body            (14300, 22)
bp              (11656, 12)
liver           (10409, 13)
hdl             (12198, 3)
triglycerides   (5090, 10)
glucose         (5090, 4)
diabetes        (14986, 28)


<h2 style="color:black;"> Renaming NHANES variables for readability </h2>

The NHANES datasets use abbreviated variable codes that make it difficult to interpret. The `rename_map` dictionary maps the NHANES codes for selected variables to easily understandable descriptive names that indicate their clinical meaning and measurement units, where applicable. This mapping is applied to every DataFrame stored in the `data` dictionary. 

For each DataFrame, columns matching a code in `rename_map` are renamed. Mapping entries that are not present in the particular DataFrame are ignored, while existing columns not listed in `rename_map` retain their original names. 

In [4]:
# Rename NHANES variables to human-readable names

rename_map = {
    # Demographics
    "RIDAGEYR": "age",                  # Age in years
    "RIAGENDR": "sex",                  # Sex
    "RIDRETH3": "race_ethnicity",       # Race/ethnicity category

    # Body measurements
    "BMXBMI": "bmi",                    # Body mass index (kg/m^2)
    "BMXWAIST": "waist_cm",             # Waist circumference (cm)

    # Blood pressure
    "BPXOSY1": "systolic_bp_1",         # Systolic BP, reading 1 (mmHg)
    "BPXOSY2": "systolic_bp_2",         # Systolic BP, reading 2 (mmHg)
    "BPXOSY3": "systolic_bp_3",         # Systolic BP, reading 3 (mmHg)
    "BPXODI1": "diastolic_bp_1",        # Diastolic BP, reading 1 (mmHg)
    "BPXODI2": "diastolic_bp_2",        # Diastolic BP, reading 2 (mmHg)
    "BPXODI3": "diastolic_bp_3",        # Diastolic BP, reading 3 (mmHg)

    # Liver elastography
    "LUAXSTAT": "elastography_status",
    "LUXCAPM": "cap_db_m",              # Median CAP; liver fat measure (dB/m)
    "LUXCPIQR": "cap_iqr",              # CAP interquartile range

    # Laboratory measurements
    "LBDHDD": "hdl_mg_dl",              # HDL cholesterol (mg/dL)
    "LBXTR": "triglycerides_mg_dl",     # Triglycerides (mg/dL)
    "LBXGLU": "glucose_mg_dl",          # Fasting glucose (mg/dL)

    # Diabetes
    "DIQ010": "diabetes_status",         # Self-reported diabetes status

    # Survey design
    "WTSAFPRP": "fasting_weight",        # Fasting subsample survey weight
}

for name in data:
    data[name] = data[name].rename(columns=rename_map)

<h2 style="color:black;"> Selecting variables for analysis </h2>

Relevant variables are selected from each component DataFrame, together with the participant identifier `SEQN`. Separate DataFrames are created for analysis using `.copy()` without modifying the original DataFrames stored in the `data` dictionary. 

Mean systolic and diastolic blood pressure are calculated using all available readings for each participant. The individual blood-pressure readings are subsequently removed from the `bp` DataFrame, leaving `SEQN` and the two derived mean measurements for subsequent merging.

In [5]:
demo = data["demo"][
    ["SEQN", "age", "sex", "race_ethnicity"]
].copy()

body = data["body"][
    ["SEQN", "bmi", "waist_cm"]
].copy()

liver = data["liver"][
    ["SEQN", "elastography_status", "cap_db_m", "cap_iqr"]
].copy()

hdl = data["hdl"][
    ["SEQN", "hdl_mg_dl"]
].copy()

triglycerides = data["triglycerides"][
    ["SEQN", "triglycerides_mg_dl", "fasting_weight"]
].copy()

glucose = data["glucose"][
    ["SEQN", "glucose_mg_dl"]
].copy()

diabetes = data["diabetes"][
    ["SEQN", "diabetes_status"]
].copy()

bp = data["bp"][
    [
        "SEQN",
        "systolic_bp_1", "systolic_bp_2", "systolic_bp_3",
        "diastolic_bp_1", "diastolic_bp_2", "diastolic_bp_3",
    ]
].copy()

bp["systolic_bp"] = bp[
    ["systolic_bp_1", "systolic_bp_2", "systolic_bp_3"]
].mean(axis=1)

bp["diastolic_bp"] = bp[
    ["diastolic_bp_1", "diastolic_bp_2", "diastolic_bp_3"]
].mean(axis=1)

bp = bp[["SEQN", "systolic_bp", "diastolic_bp"]]

<h2 style="color:black;"> Checking uniqueness of participant identifier </h2>

The selected component DataFrames are stored in the `datasets` dictionary. The total number of rows, unique `SEQN` identifiers and duplicated identifiers are displayed for each DataFrame to assess participant-identifier uniqueness before merging.

In [6]:
datasets = {
    "demo": demo,
    "body": body,
    "bp": bp,
    "liver": liver,
    "hdl": hdl,
    "triglycerides": triglycerides,
    "glucose": glucose,
    "diabetes": diabetes,
}

for name, dataset in datasets.items():
    print(
        f"{name:15} "
        f"rows={len(dataset):6,} | "
        f"unique SEQN={dataset['SEQN'].nunique():6,} | "
        f"duplicates={dataset['SEQN'].duplicated().sum()}"
    )

demo            rows=15,560 | unique SEQN=15,560 | duplicates=0
body            rows=14,300 | unique SEQN=14,300 | duplicates=0
bp              rows=11,656 | unique SEQN=11,656 | duplicates=0
liver           rows=10,409 | unique SEQN=10,409 | duplicates=0
hdl             rows=12,198 | unique SEQN=12,198 | duplicates=0
triglycerides   rows= 5,090 | unique SEQN= 5,090 | duplicates=0
glucose         rows= 5,090 | unique SEQN= 5,090 | duplicates=0
diabetes        rows=14,986 | unique SEQN=14,986 | duplicates=0


All component datasets contain zero duplicated `SEQN` identifiers. This confirms that each dataset is suitable for one-to-one merging on `SEQN`.

<h2 style="color:black;"> Merging the component datasets </h2>

The component DataFrames are sequentially merged on the participant identifier `SEQN` using left joins, starting with the demographics DataFrame `demo`. This preserves every participant represented in `demo` while adding variables from the remaining component DataFrames for participants with matching `SEQN` identifiers. 

The `validate="one_to_one"` argument verifies that each merge maintains the expected one-to-one relationship between participant records. 

The dimensions and first five rows of the merged DataFrame `df` are displayed to verify the result. 

In [7]:
from functools import reduce

df = reduce(
    lambda left, right: left.merge(
        right,
        on="SEQN",
        how="left",
        validate="one_to_one"
    ),
    datasets.values()
)

print(df.shape)
df.head()

(15560, 16)


,SEQN,age,sex,race_ethnicity,bmi,waist_cm,systolic_bp,diastolic_bp,elastography_status,cap_db_m,cap_iqr,hdl_mg_dl,triglycerides_mg_dl,fasting_weight,glucose_mg_dl,diabetes_status
0,109263.0,2.0,1.0,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
1,109264.0,13.0,2.0,1.0,17.6,63.8,108.0,67.000000,3.0,NaN,NaN,72.0,40.0,27533.174559,97.0,2.0
2,109265.0,2.0,1.0,3.0,15.0,41.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
3,109266.0,29.0,2.0,6.0,37.8,117.9,99.0,54.333333,1.0,277.0,45.0,56.0,NaN,NaN,NaN,2.0
4,109267.0,21.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0


Missing values, represented as `NaN`, may occur when a participant in the demographics dataset is not represented in a particular component dataset, or when an individual examination, laboratory or questionnaire value is unavailable.

The merged DataFrame contains 15,560 rows representing participants and 16 columns representing variables. This row count matches the `demo` DataFrame, confirming that the sequential left joins retained all demographic records as intended.

<h2 style="color:black;"> Initial cohort filtering and data quality assessment </h2>

<h3 style="color:black;"> Applying eligibility criteria </h3>

Although NHANES includes participants across a broad age range (from 2 months upwards), this study focuses on adults aged 18 years or older. This age restriction is applied to the merged DataFrame `df`, and participant counts before and after filtering are displayed to document cohort attrition. 

In [8]:
n_all = len(df)

df = df[df["age"] >= 18].copy()

print(f"All participants: {n_all:,}")
print(f"Adults 18+:       {len(df):,}")
print(f"Removed:          {n_all - len(df):,}")

All participants: 15,560
Adults 18+:       9,693
Removed:          5,867


<h3 style="color:black;"> Assessing initial missingness </h3> 

The number and percentage of missing values are calculated for each variable in the adult cohort and sorted in descending order of missingness. This provides an initial overview of data availability before the remaining eligibility criteria are applied. 

Because the merged DataFrame retains all adult participants from the demographics dataset, some values may be missing because participants were ineligible for or did not participate in the fasting laboratory or elastography component, rather than because an individual response or measurement was omitted.

In [9]:
missing = pd.DataFrame({
    "missing_n": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1)
})

missing.sort_values("missing_pct", ascending=False)

,missing_n,missing_pct
triglycerides_mg_dl,5593,57.7
glucose_mg_dl,5520,56.9
fasting_weight,5255,54.2
systolic_bp,1669,17.2
diastolic_bp,1669,17.2
hdl_mg_dl,1395,14.4
cap_iqr,1383,14.3
cap_db_m,1376,14.2
waist_cm,1244,12.8
bmi,903,9.3


<h3 style="color:black;"> Inspecting categorical variable distributions </h3> 

The frequency distributions of `sex`, `race_ethnicity` and `diabetes_status` are displayed. Existing `NaN` values are included in the counts, while sorting by category code facilitates comparison with the NHANES codebooks for interpretation. 

This comparison identifies the special response code `9` (“Don’t know”) for the `diabetes_status` variable. This code is currently stored as a numeric value but should be recoded as missing before analysis.

In [10]:
for column in ["sex", "race_ethnicity", "diabetes_status"]:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False).sort_index())


--- sex ---
sex
1.0    4718
2.0    4975
Name: count, dtype: int64

--- race_ethnicity ---
race_ethnicity
1.0    1131
2.0     991
3.0    3370
4.0    2555
6.0    1172
7.0     474
Name: count, dtype: int64

--- diabetes_status ---
diabetes_status
1.0    1423
2.0    7998
3.0     267
9.0       5
Name: count, dtype: int64


<h3 style="color:black;"> Summarising numerical variables </h3>

Descriptive statistics are generated for the selected numerical variables to examine their distributions and identify potentially unusual values before further processing. 

The summary displays the number of non-missing observations (`count`), mean, standard deviation, minimum, quartiles and maximum for each variable. The table is transposed (`.T`) so that variables are displayed as rows for easier comparison.

In [11]:
numeric_cols = [
    "age",
    "bmi",
    "waist_cm",
    "systolic_bp",
    "diastolic_bp",
    "cap_db_m",
    "cap_iqr",
    "hdl_mg_dl",
    "triglycerides_mg_dl",
    "glucose_mg_dl",
]

df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
age,9693.0,49.591974,18.607845,1.800000e+01,33.000000,50.000000,64.000000,80.000000
bmi,8790.0,29.883413,7.603916,1.420000e+01,24.700000,28.600000,33.600000,92.300000
waist_cm,8449.0,100.397574,17.349142,5.640000e+01,88.100000,99.000000,111.000000,187.500000
systolic_bp,8024.0,124.219570,19.324974,7.633333e+01,110.333333,121.333333,134.666667,218.666667
diastolic_bp,8024.0,74.345339,11.623023,4.133333e+01,66.333333,73.666667,81.333333,143.666667
cap_db_m,8317.0,263.725141,63.084534,1.000000e+02,217.000000,262.000000,308.000000,400.000000
cap_iqr,8310.0,37.416005,20.218826,5.397605e-79,24.000000,34.000000,47.000000,192.000000
hdl_mg_dl,8298.0,53.455893,15.859462,5.000000e+00,42.000000,51.000000,62.000000,189.000000
triglycerides_mg_dl,4100.0,108.253659,93.442882,1.000000e+01,59.000000,88.000000,131.000000,2684.000000
glucose_mg_dl,4173.0,112.913012,37.525950,4.700000e+01,95.000000,103.000000,114.000000,451.000000


<h3 style="color:black;"> Inspecting liver elastography examination status </h3>

The frequency distribution of the `elastography_status` variable is displayed to assess the status of participants' liver transient elastography examinations. Existing `NaN` values after merging are included and indicate that no matching elastography record was added for the participant during the left merge. 

The categories are sorted by status code for easy comparison with the NHANES codebook. This assessment informs the subsequent selection of participants with complete elastography examinations.

According to the NHANES codebook:
- `1`: Complete examination 
- `2`: Partial examination 
- `3`: Ineligible 
- `4`: Not done 

In [12]:
df["elastography_status"].value_counts(dropna=False).sort_index()

elastography_status
1.0    7768
2.0     616
3.0     348
4.0     233
NaN     728
Name: count, dtype: int64

<h3 style="color:black;"> Applying elastography criteria and recoding categorical variables </h3>

The adult cohort is restricted to participants with a complete liver transient elastography examination (`elastography_status == 1`) and an available median controlled attenuation parameter (CAP) measurement (`cap_db_m`). Participant counts before and after filtering are displayed to document cohort attrition. 

The numeric NHANES codes for `sex`, `race_ethnicity` and `diabetes_status` are subsequently mapped to descriptive category labels. Special non-response codes as previously mentioned are recoded as missing. The distributions of the recoded variables are displayed to verify the transformations. 

In [13]:
# Keep eligible adults with a complete elastography exam and available CAP

n_adults = len(df)

df = df[
    (df["elastography_status"] == 1) &
    (df["cap_db_m"].notna())
].copy()

# Convert NHANES categorical codes to readable values
df["sex"] = df["sex"].map({
    1: "Male",
    2: "Female",
})

df["race_ethnicity"] = df["race_ethnicity"].map({
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian",
    7: "Other / Multiracial",
})

df["diabetes_status"] = df["diabetes_status"].map({
    1: "Yes",
    2: "No",
    3: "Borderline", 
    7: pd.NA,  # Refused
    9: pd.NA,  # Don't know
})

print(f"Adults 18+:                    {n_adults:,}")
print(f"Complete elastography + CAP:   {len(df):,}")
print(f"Excluded:                      {n_adults - len(df):,}")

print("\nElastography status:")
print(df["elastography_status"].value_counts(dropna=False))

print("\nCategorical variables:")
for col in ["sex", "race_ethnicity", "diabetes_status"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

Adults 18+:                    9,693
Complete elastography + CAP:   7,767
Excluded:                      1,926

Elastography status:
elastography_status
1.0    7767
Name: count, dtype: int64

Categorical variables:

sex:
sex
Female    3906
Male      3861
Name: count, dtype: int64

race_ethnicity:
race_ethnicity
Non-Hispanic White     2637
Non-Hispanic Black     2040
Non-Hispanic Asian      943
Mexican American        943
Other Hispanic          810
Other / Multiracial     394
Name: count, dtype: int64

diabetes_status:
diabetes_status
No            6457
Yes           1082
Borderline     224
NaN              4
Name: count, dtype: int64


<h3 style="color:black;"> Assessing missingness and predictor completeness </h3>

The number and percentage of missing values are recalculated after restricting the cohort by age, elastography status and CAP availability. Variables are sorted in descending order of missingness to identify those that may require further treatment (e.g. imputation) or additional eligibility restrictions. 

Completeness at the participant level is then assessed for two predictor sets. The core set contains anthropometric, blood pressure and HDL measurements, while the expanded set additionally contains fasting triglycerides and glucose measurements. These counts show the potential sample sizes available under different predictor inclusion requirements.

In [14]:
# Examine missing data in the eligible study cohort

missing = pd.DataFrame({
    "missing_n": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(1),
}).sort_values("missing_pct", ascending=False)

print(f"Eligible cohort: {len(df):,} participants")
display(missing)

# How many participants would remain under different predictor requirements?
core_predictors = [
    "bmi",
    "waist_cm",
    "systolic_bp",
    "diastolic_bp",
    "hdl_mg_dl",
]

fasting_predictors = [
    "triglycerides_mg_dl",
    "glucose_mg_dl",
]

print(
    "\nComplete core predictors:",
    df[core_predictors].notna().all(axis=1).sum()
)

print(
    "Complete core + fasting predictors:",
    df[core_predictors + fasting_predictors].notna().all(axis=1).sum()
)

Eligible cohort: 7,767 participants


,missing_n,missing_pct
triglycerides_mg_dl,4117,53.0
glucose_mg_dl,4056,52.2
fasting_weight,3853,49.6
systolic_bp,676,8.7
diastolic_bp,676,8.7
hdl_mg_dl,483,6.2
waist_cm,251,3.2
bmi,69,0.9
diabetes_status,4,0.1
SEQN,0,0.0



Complete core predictors: 6546
Complete core + fasting predictors: 3272


**Note:** The preliminary counts above consider only the two metabolic predictor sets. Completeness of the demographic and clinical predictors is additionally assessed when constructing the final analysis cohort.

<h2 style="color:black;"> Constructing the complete-case analysis cohort </h2>

The predictor variables required for the analysis are classified into metabolic, demographic and clinical feature sets. The metabolic set represents anthropometric characteristics, blood pressure, lipid metabolism and glucose regulation. The demographic set contains age, sex and race/ethnicity, while the clinical set contains self-reported diabetes status. Participants with one or more missing values among these required predictors are excluded to form the complete-case analysis cohort.

In [15]:
# Define variables required for analysis

# Fasting-subsample eligibility is applied before complete-case selection
# to distinguish design-based exclusions from missing predictor data.

metabolic_features = [
    "bmi",
    "waist_cm",
    "systolic_bp",
    "diastolic_bp",
    "hdl_mg_dl",
    "triglycerides_mg_dl",
    "glucose_mg_dl",
]

demographic_features = [
    "age",
    "sex",
    "race_ethnicity",
]

clinical_features = [
    "diabetes_status",
]

required_columns = metabolic_features + demographic_features + clinical_features

# Restrict the eligible liver cohort to the fasting subsample
n_liver_cohort = len(df)

fasting_df = df[
    df["fasting_weight"].notna() &
    (df["fasting_weight"] > 0)
].copy()

# Retain complete cases among fasting-eligible participants
analysis_df = fasting_df.dropna(subset=required_columns).copy()

print(f"Eligible liver cohort:                  {n_liver_cohort:,}")
print(f"Eligible fasting subsample:             {len(fasting_df):,}")
print(
    f"Not included in fasting subsample:      "
    f"{n_liver_cohort - len(fasting_df):,}"
)
print(f"Complete analysis cohort:               {len(analysis_df):,}")
print(
    f"Excluded for missing predictor values:  "
    f"{len(fasting_df) - len(analysis_df):,}"
)
print(
    f"Total excluded from liver cohort:       "
    f"{n_liver_cohort - len(analysis_df):,}"
)

print("\nRemaining missing values:")
print(analysis_df[required_columns].isna().sum())

Eligible liver cohort:                  7,767
Eligible fasting subsample:             3,914
Not included in fasting subsample:      3,853
Complete analysis cohort:               3,270
Excluded for missing predictor values:  644
Total excluded from liver cohort:       4,497

Remaining missing values:
bmi                    0
waist_cm               0
systolic_bp            0
diastolic_bp           0
hdl_mg_dl              0
triglycerides_mg_dl    0
glucose_mg_dl          0
age                    0
sex                    0
race_ethnicity         0
diabetes_status        0
dtype: int64


Of the 7,767 participants in the eligible liver cohort, 3,853 were not included in the fasting subsample. Restricting the cohort to participants with complete data for all required predictors excluded a further 644 participants, resulting in a final analysis cohort of 3,270 participants. Therefore, most exclusions were attributable to the NHANES fasting-subsample design rather than missing predictor values among fasting-subsample participants.

<h2 style="color:black;"> Defining the hepatic steatosis outcome </h2>

Hepatic steatosis is defined using the median controlled attenuation parameter (`cap_db_m`) obtained from liver transient elastography. A threshold of 248 dB/m was reported in an individual-participant-data meta-analysis (Karlas et al., 2017) and subsequently applied to NHANES 2017–2020 data (Xi & Yang, 2024). Based on this literature-derived threshold, participants with a CAP measurement at or above 248 dB/m are classified as having hepatic steatosis (`hepatic_steatosis = 1`), while those below it are classified as not having hepatic steatosis (`hepatic_steatosis = 0`).

The number of participants in each outcome class and the prevalence of hepatic steatosis in the final analysis cohort are displayed to assess the outcome distribution.

In [16]:
# Define hepatic steatosis using literature-derived CAP threshold

CAP_CUTOFF = 248

analysis_df["hepatic_steatosis"] = (
    analysis_df["cap_db_m"] >= CAP_CUTOFF
).astype(int)

counts = analysis_df["hepatic_steatosis"].value_counts().sort_index()

print(f"No steatosis: {counts.get(0, 0):,}")
print(f"Steatosis:    {counts.get(1, 0):,}")
print(f"Prevalence:   {analysis_df['hepatic_steatosis'].mean() * 100:.1f}%")

No steatosis: 1,388
Steatosis:    1,882
Prevalence:   57.6%


<h2 style="color:black;"> Finalising and saving the analysis cohort </h2>

The prepared dataset contains only the participant identifier, selected metabolic, demographic and clinical predictors, median CAP measurement and binary hepatic steatosis outcome for downstream analyses.

Final validation checks are performed using `assert` statements. The dimensions and outcome distribution of the final analysis cohort are displayed.

Although `cap_db_m` is retained for descriptive reporting, it will not be used as a predictor since the hepatic steatosis outcome is derived directly from this measurement. This prevents data leakage during supervised modelling.

In [17]:
# Final checks and save prepared analysis cohort

# Columns needed for downstream analysis
final_columns = [
    "SEQN",
    "age",
    "sex",
    "race_ethnicity",
    "bmi",
    "waist_cm",
    "systolic_bp",
    "diastolic_bp",
    "hdl_mg_dl",
    "triglycerides_mg_dl",
    "glucose_mg_dl",
    "diabetes_status",
    "cap_db_m",
    "hepatic_steatosis",
]

analysis_df = analysis_df[final_columns].copy()

# Sanity checks
assert analysis_df["SEQN"].is_unique
assert analysis_df.isna().sum().sum() == 0
assert set(analysis_df["hepatic_steatosis"].unique()) == {0, 1}

print(f"Final cohort: {len(analysis_df):,} participants")
print(f"Variables:    {analysis_df.shape[1]}")
print("\nOutcome:")
print(analysis_df["hepatic_steatosis"].value_counts())

# Save
output_path = PROCESSED_DIR / "analysis_cohort.csv"
analysis_df.to_csv(output_path, index=False)

print(f"\nSaved to: {output_path}")

Final cohort: 3,270 participants
Variables:    14

Outcome:
hepatic_steatosis
1    1882
0    1388
Name: count, dtype: int64

Saved to: ../data/processed/analysis_cohort.csv


The final validation checks confirm that each participant identifier in the cohort is unique, no selected variables contain missing values and both hepatic steatosis outcome classes are represented. The final analysis cohort is saved as `analysis_cohort.csv` for downstream analyses.

<h2 style="color:black;"> References </h2>

1. Karlas, T., Petroff, D., Sasso, M., Fan, J.-G., Mi, Y.-Q., Lédinghen, V. de, Kumar, M., Lupsor-Platon, M., Han, K.-H., Cardoso, A. C., Ferraioli, G., Chan, W.-K., Wong, V. W.-S., Myers, R. P., Chayama, K., Friedrich-Rust, M., Beaugrand, M., Shen, F., Hiriart, J.-B., … Wiegand, J. (2017). Individual patient data meta-analysis of controlled attenuation parameter (CAP) technology for assessing steatosis. Journal of Hepatology, 66(5), 1022–1030. https://doi.org/10.1016/j.jhep.2016.12.022

2. Xi, W., & Yang, A. (2024). Association between cardiometabolic index and controlled attenuation parameter in U.S. adults with NAFLD: Findings from NHANES (2017–2020). Lipids in Health and Disease, 23(1), 40. https://doi.org/10.1186/s12944-024-02027-x